# Fermionic h=0 architecture ladder — per-subplot builder

Three tiers at $h_x=h_z=0$: **CNN** (GeoCNN, symmetry-unaware, complex) vs
**Approx. Symm.** (`ToricCNN_gridinv`, no sign head, cold) vs **Sign-Head +
Approx. Symm.** (frozen analytic GF(2) head + flux penalty + `chains_up`).
Exact anchor $E_0=-4L^3$. Data: `results/fermionic_ladder/*.curve.json`
(campaign 2026-08-19, `nersc/launch_fermionic_ladder.sh`).

**Workflow:** run §1–§2, read the roster, edit the knobs in §1
(`INCLUDE_PATTERNS` / `EXCLUDE_RUNS` / `EXCLUDE_PREFIXES` /
`AUTO_SKIP_DIVERGED`), re-run §2, then build any subplot cell — each panel is
its own figure with its own (commented) `savefig`. The composite grid at the
end uses the same selection.

In [ ]:
# %% 1. CONFIG — the run-selection knobs live here ---------------------------
import json, glob, os, re, fnmatch
import numpy as np
import matplotlib.pyplot as plt

BASE   = "../results/fermionic_ladder"
FIGS   = "figs"
LS     = [2, 3, 4]
E0     = {L: -4 * L**3 for L in LS}          # exact h=0 PBC anchor
NSPIN  = {L: 3 * L**3 for L in LS}
TRAP_L2 = -22.521                            # positive-sector optimum (BLOG 2026-08-07)

# tier -> (label, color); tweak labels for the talk here
TIERS = {
    "cnn":      ("CNN",        plt.cm.plasma(0.08)),
    "asymm":    ("Approx. Symm.",             plt.cm.plasma(0.55)),
    "signhead": ("Sign-aware",                plt.cm.plasma(0.85)),
}
VARIANT_LS = {"ch444": "-", "ch88": "--", "ch88g": "--", "inv88": "-",
              "inv222": "--", "inv222g": "--", "hist": "-", "": "-"}

# Historical analytic-head polishes (BLOG 2026-08-07) — the "learning-visible"
# sign-head examples: amplitudes started far from uniform (raw prefit container),
# signs exact throughout, so the visible descent is pure amplitude learning.
# label -> (path, (tier, L, variant, seed)); variant "hist" renders dash-dot.
EXTRA_CURVES = {
    "signhead_hist_polishANA_L2":
        ("../results/fermionic_h0/gridinv_fermionic_L2_PBC_h0_n2x4_nh4-8_inv8-8_k2_ph_polishANA.curve.json",
         ("signhead", 2, "hist", 0)),
    "signhead_hist_polishANA_L3":
        ("../results/fermionic_h0/gridinv_fermionic_L3_PBC_h0_n2x4_nh4-8_inv8-8_k2_ph_fp6_polishANA.curve.json",
         ("signhead", 3, "hist", 0)),
}
ERR_FLOOR = 1e-9                             # log-panel clip (sampling floor)
YLIM_E    = (-1.45, 0.02)                    # uniform E/N scale; clips rollback transients

# ---- WHICH runs to include --------------------------------------------------
# Every discovered run is listed in the roster (next cell) with its final E,
# Im<E>, and flags, plus the verdict of these knobs. Adjust and re-run.
AUTO_SKIP_DIVERGED = True                    # skip runs banked diverged=True in their final .json
INCLUDE_PATTERNS = ["ladder_cnn_*", "ladder_asymm_*", "ladder_signhead_L4_*"]   # fnmatch globs.
# Campaign signhead runs (born-converged, flat) are pattern-excluded: the Sign-aware
# tier is represented by the amp-warm EXTRA_CURVES below, which show the descent.
# Restore the flat runs with e.g. "ladder_signhead_*" (or just "ladder_signhead_L4_*").
EXCLUDE_RUNS = {                             # exact run names to drop (your call, by name)
    # e.g. "ladder_asymm_L3_inv88_s1",       # the seed stuck at the -54 attractor
}
EXCLUDE_PREFIXES = (                         # opened-guard runs superseded by the *g stock-guard reruns
    "ladder_cnn_L2_ch88_",                   # 3/3 collapse (spread blow-up, no guard net)
    "ladder_asymm_L3_inv222_",               # sign-corrupted finals (Im<E> ~ 5)
)

plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# %% 2. load EVERYTHING + roster --------------------------------------------
# name convention: ladder_{tier}_L{L}_{variant}_s{seed}[_smoke]
PAT = re.compile(r"ladder_(?P<tier>cnn|asymm|signhead)_L(?P<L>\d)(?:_(?P<var>[a-z0-9]+))?_s(?P<seed>\d+)")

def _verdict(name, diverged):
    if name.endswith("_smoke"):                          return "smoke"
    if any(name.startswith(p) for p in EXCLUDE_PREFIXES): return "prefix-excluded"
    if AUTO_SKIP_DIVERGED and diverged:                  return "diverged"
    if name in EXCLUDE_RUNS:                             return "excluded by name"
    if not any(fnmatch.fnmatch(name, p) for p in INCLUDE_PATTERNS): return "no pattern match"
    return "INCLUDED"

all_runs = {}                                 # name -> dict(step, E, imE, diverged, verdict, key)
for path in sorted(glob.glob(os.path.join(BASE, "ladder_*.curve.json"))):
    name = os.path.basename(path)[:-len(".curve.json")]
    m = PAT.match(name)
    if not m:
        print("unparsed name, skipping:", name); continue
    d = json.load(open(path))["curve"]
    step = np.asarray(d["step"], float); E = np.asarray(d["energy"], float)
    im = np.asarray(d.get("energy_im") or np.zeros_like(E), float)
    o = np.argsort(step); step, E, im = step[o], E[o], im[o]
    keep = np.concatenate([[True], np.diff(step) > 0]); step, E, im = step[keep], E[keep], im[keep]
    fj = path.replace(".curve.json", ".json")
    diverged = bool(os.path.exists(fj) and json.load(open(fj)).get("diverged"))
    all_runs[name] = {"step": step, "E": E, "imE": im, "diverged": diverged,
                      "key": (m["tier"], int(m["L"]), m["var"] or "", int(m["seed"])),
                      "verdict": _verdict(name, diverged)}

for label, (path, key) in EXTRA_CURVES.items():
    if not os.path.exists(path):
        print("extra curve missing, skipping:", path); continue
    d = json.load(open(path))["curve"]
    E = np.asarray(d["energy"], float)
    im = np.asarray(d.get("energy_im") or np.zeros_like(E), float)
    all_runs[label] = {"step": np.asarray(d["step"], float), "E": E, "imE": im,
                       "diverged": False, "key": key,
                       "verdict": "excluded by name" if label in EXCLUDE_RUNS else "INCLUDED"}

runs = {n: r for n, r in all_runs.items() if r["verdict"] == "INCLUDED"}

print(f"{'run':42} {'steps':>5} {'final E':>13} {'rel.err':>9} {'|imE|':>8} {'verdict'}")
for n, r in sorted(all_runs.items(), key=lambda kv: kv[1]["key"]):
    L = r["key"][1]; e = r["E"][-1]
    print(f"{n:42} {int(r['step'][-1]):>5} {e:>13.6f} {abs(e-E0[L])/abs(E0[L]):>9.2e} "
          f"{abs(r['imE'][-1]):>8.1e} {r['verdict']}")
print(f"\n{len(runs)} of {len(all_runs)} runs included")

In [ ]:
# %% 3. panel helpers ---------------------------------------------------------
from matplotlib.lines import Line2D

def _legend_order(h, l):
    rank = {"CNN": 0, "Approx. Symm.": 1, "Sign-aware": 2}
    pairs = sorted(zip(h, l), key=lambda hl: rank.get(hl[1], 99))
    return [p[0] for p in pairs], [p[1] for p in pairs]

def panel_energy(L, ax, legend=False):
    """E/N vs SR step for one system size, included runs only."""
    seen = set()
    for n, r in sorted(runs.items(), key=lambda kv: kv[1]["key"]):
        tier, Lk, var, seed = r["key"]
        if Lk != L: continue
        label, col = TIERS[tier]
        ax.plot(r["step"], r["E"] / NSPIN[L], color=col, lw=1.4,
                ls=VARIANT_LS.get(var, "-"), alpha=0.85, zorder=3,
                label=label if tier not in seen else None)
        seen.add(tier)
    ax.axhline(E0[L] / NSPIN[L], color="k", ls="--", lw=1.2, zorder=1)
    if L == 2:
        ax.axhline(TRAP_L2 / NSPIN[2], color="grey", ls=":", lw=1.2, zorder=1)
    ax.set_ylim(*YLIM_E)
    ax.set_title(f"$L={L}$  ($N={NSPIN[L]}$ spins, $E_0=-4L^3={E0[L]}$)")
    ax.set_xlabel("SR step"); ax.set_ylabel(r"$\langle H \rangle / N$")
    if legend:
        h, l = _legend_order(*ax.get_legend_handles_labels())
        ax.legend(h, l, frameon=False, fontsize=9, loc="lower right")

def panel_relerr(L, ax):
    """log10 |E-E0|/|E0| vs SR step for one system size, included runs only."""
    for n, r in sorted(runs.items(), key=lambda kv: kv[1]["key"]):
        tier, Lk, var, seed = r["key"]
        if Lk != L: continue
        rel = np.clip(np.abs(r["E"] - E0[L]) / abs(E0[L]), ERR_FLOOR, None)
        ax.plot(r["step"], rel, color=TIERS[tier][1], lw=1.4,
                ls=VARIANT_LS.get(var, "-"), alpha=0.85, zorder=3)
    ax.set_yscale("log")
    ax.set_xlabel("SR step"); ax.set_ylabel(r"$|E - E_0| / |E_0|$")
    ax.set_title(f"$L={L}$ — relative error")

In [ ]:
# %% E/N panel, L=2 ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.6, 4.0))
panel_energy(2, ax, legend=True)
# plt.savefig(os.path.join(FIGS, "fermionic_ladder_E_L2.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# %% relative-error panel, L=2 -----------------------------------------------
fig, ax = plt.subplots(figsize=(5.6, 3.4))
panel_relerr(2, ax)
# plt.savefig(os.path.join(FIGS, "fermionic_ladder_relerr_L2.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# %% E/N panel, L=3 ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.6, 4.0))
panel_energy(3, ax, legend=False)
# plt.savefig(os.path.join(FIGS, "fermionic_ladder_E_L3.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# %% relative-error panel, L=3 -----------------------------------------------
fig, ax = plt.subplots(figsize=(5.6, 3.4))
panel_relerr(3, ax)
# plt.savefig(os.path.join(FIGS, "fermionic_ladder_relerr_L3.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# %% E/N panel, L=4 ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.6, 4.0))
panel_energy(4, ax, legend=False)
# plt.savefig(os.path.join(FIGS, "fermionic_ladder_E_L4.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# %% relative-error panel, L=4 -----------------------------------------------
fig, ax = plt.subplots(figsize=(5.6, 3.4))
panel_relerr(4, ax)
# plt.savefig(os.path.join(FIGS, "fermionic_ladder_relerr_L4.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# %% composite 2x3 grid (same include knobs) ----------------------------------
fig, axes = plt.subplots(2, len(LS), figsize=(13.5, 7.2), sharex="col",
                         gridspec_kw={"height_ratios": [1.25, 1.0], "hspace": 0.08})
for j, L in enumerate(LS):
    panel_energy(L, axes[0, j], legend=False)
    panel_relerr(L, axes[1, j])
    axes[0, j].set_xlabel(""); axes[1, j].set_title("")
    if j > 0:
        axes[0, j].set_ylabel(""); axes[1, j].set_ylabel("")
h, l = _legend_order(*axes[0, 0].get_legend_handles_labels())
fig.subplots_adjust(top=0.86)
fig.legend(h, l, ncol=3, frameon=False, fontsize=10, loc="upper center", bbox_to_anchor=(0.5, 0.94))
fig.suptitle("fermionic 3D toric code, $h=0$ — what the sign head buys, across architectures", y=0.99)
# plt.savefig(os.path.join(FIGS, "fermionic_arch_ladder.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# %% plateau table (included runs only) ---------------------------------------
print(f"{'tier':10} {'L':>2} {'variant':8} {'seed':>4} {'E (last-20 mean)':>18} {'rel. err':>10}")
for n, r in sorted(runs.items(), key=lambda kv: kv[1]["key"]):
    tier, L, var, seed = r["key"]
    e = float(np.mean(r["E"][-20:]))
    print(f"{tier:10} {L:>2} {var or '-':8} {seed:>4} {e:>18.6f} {abs(e-E0[L])/abs(E0[L]):>10.2e}")